In [17]:
import numpy as np
BASE_PATH = '../../assets/raw'

def load_image(file):
    with open(file, 'rb') as f:
        f.seek(16)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data.reshape(-1, 1, 28, 28)  # Reshape to (num_images, channels, height, width)

def load_label(file):
    with open(file, 'rb') as f:
        f.seek(8)  # Skip the header
        data = np.frombuffer(f.read(), dtype=np.uint8)
    return data

In [18]:
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]

In [19]:
def relu(Z):
    return np.maximum(0, Z)

In [20]:
def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True)) # sub max for numerical stability
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [21]:
def relu_derivative(Z):
    return (Z > 0).astype(float) # derivative of ReLU (1 if Z > 0 else 0)

In [22]:
def output_shape(input_shape, kernel_size, stride, padding):
    return (input_shape - kernel_size + 2 * padding) // stride + 1

In [23]:
def split_mini_batches(X, Y, batch_size):
    """X: (N, C, H, W).  Y: (num_classes, N)."""
    m = X.shape[0]
    perm = np.random.permutation(m)
    X_shuf, Y_shuf = X[perm], Y[:, perm]
    return [(X_shuf[i:i + batch_size], Y_shuf[:, i:i + batch_size])
               for i in range(0, m, batch_size)]

In [24]:
def conv_step(a_slice_prev, W, b):
    return np.sum(a_slice_prev * W) + b.item()

In [25]:
def conv_forward(A_prev, W, b, stride, padding):
    (m, n_C_prev, n_H_prev, n_W_prev) = A_prev.shape
    (n_C, _, f, _) = W.shape

    # output dimensions
    n_H = output_shape(n_H_prev, f, stride, padding)
    n_W = output_shape(n_W_prev, f, stride, padding)
    Z = np.zeros((m, n_C, n_H, n_W))

    A_prev_pad = np.pad(A_prev, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')

    for i in range(m):
        a_prev_pad = A_prev_pad[i]
        for c in range(n_C):
            for h in range(n_H):
                for w in range(n_W):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f

                    a_slice = a_prev_pad[:, vert_start:vert_end, horiz_start:horiz_end]
                    Z[i, c, h, w] = conv_step(a_slice, W[c], b[c])

    cache = (A_prev, W, b, stride, padding)
    return Z, cache

In [26]:
def pool_forward(A_prev, f, stride, mode='max'):
    (m, n_C, n_H_prev, n_W_prev) = A_prev.shape

    # output dimensions
    n_H = output_shape(n_H_prev, f, stride, 0)
    n_W = output_shape(n_W_prev, f, stride, 0)
    A = np.zeros((m, n_C, n_H, n_W))

    for i in range(m):
        for c in range(n_C):
            for h in range(n_H):
                for w in range(n_W):
                    vert_start = h * stride
                    vert_end = vert_start + f
                    horiz_start = w * stride
                    horiz_end = horiz_start + f

                    a_slice = A_prev[i, c, vert_start:vert_end, horiz_start:horiz_end]

                    if mode == 'max':
                        A[i, c, h, w] = np.max(a_slice)
                    elif mode == 'average':
                        A[i, c, h, w] = np.mean(a_slice)

    cache = (A_prev, f, stride, mode)
    return A, cache

In [27]:
def forward_pass(X, params, conv_layers, fc_layers):
    cache = {}
    A = X  # (N, C, H, W)

    for i, layer in enumerate(conv_layers, start=1):
        W, b = params[f'W{i}'], params[f'b{i}']
        Z, conv_cache = conv_forward(A, W, b, layer['s'], layer['p'])

        if layer['activation'] == 'relu':
            A_act = relu(Z)
        elif layer['activation'] == 'softmax':
            A_act = softmax(Z)
        else:
            A_act = Z

        pool = layer['pool']
        A, pool_cache = pool_forward(A_act, pool['f'], pool['s'], mode=pool['type'])

        cache[f'conv{i}'] = {'Z': Z, 'conv_cache': conv_cache, 'pool_cache': pool_cache}

    cache['flatten_shape'] = A.shape          # (N, C, H, W) for backward un-flatten
    A = A.reshape(A.shape[0], -1).T           # (features, N)

    offset = len(conv_layers)
    for i, layer in enumerate(fc_layers, start=1):
        W, b = params[f'W{offset + i}'], params[f'b{offset + i}']
        A_prev = A
        Z = W @ A_prev + b

        if layer['activation'] == 'relu':
            A = relu(Z)
        elif layer['activation'] == 'softmax':
            A = softmax(Z)
        else:
            A = Z

        cache[f'fc{i}'] = {'A_prev': A_prev, 'Z': Z, 'W': W}

    return A, cache     # A shape: (num_classes, N)

In [28]:
def cross_entropy_loss(AL, Y, eps=1e-12):
    m = Y.shape[1]
    return -np.sum(Y * np.log(AL + eps)) / m

In [29]:
def conv_backward(dZ, cache):
    A_prev, W, b, stride, padding = cache
    m, n_C_prev, n_H_prev, n_W_prev = A_prev.shape
    n_C, _, f, _ = W.shape
    _, _, n_H, n_W = dZ.shape

    dW = np.zeros_like(W)
    db = np.zeros_like(b)

    A_prev_pad  = np.pad(A_prev, ((0,0),(0,0),(padding,padding),(padding,padding)), mode='constant')
    dA_prev_pad = np.zeros_like(A_prev_pad)

    for i in range(m):
        for c in range(n_C):
            for h in range(n_H):
                for w in range(n_W):
                    vs, ve = h * stride, h * stride + f
                    hs, he = w * stride, w * stride + f
                    a_slice = A_prev_pad[i, :, vs:ve, hs:he]

                    dA_prev_pad[i, :, vs:ve, hs:he] += W[c] * dZ[i, c, h, w]
                    dW[c] += a_slice * dZ[i, c, h, w]
                    db[c] += dZ[i, c, h, w]

    if padding > 0:
        dA_prev = dA_prev_pad[:, :, padding:-padding, padding:-padding]
    else:
        dA_prev = dA_prev_pad
    return dA_prev, dW, db

In [30]:
def pool_backward(dA, cache):
    A_prev, f, stride, mode = cache
    m, n_C, n_H, n_W = dA.shape
    dA_prev = np.zeros_like(A_prev)

    for i in range(m):
        for c in range(n_C):
            for h in range(n_H):
                for w in range(n_W):
                    vs, ve = h * stride, h * stride + f
                    hs, he = w * stride, w * stride + f
                    if mode == 'max':
                        a_slice = A_prev[i, c, vs:ve, hs:he]
                        mask = (a_slice == np.max(a_slice))
                        dA_prev[i, c, vs:ve, hs:he] += mask * dA[i, c, h, w]
                    elif mode == 'average':
                        dA_prev[i, c, vs:ve, hs:he] += dA[i, c, h, w] / (f * f)
    return dA_prev

In [31]:
def backward_pass(AL, Y, cache, conv_layers, fc_layers):
    grads = {}
    m = Y.shape[1]
    n_conv, n_fc = len(conv_layers), len(fc_layers)

    dZ = (AL - Y) / m
    dA = None

    for i in reversed(range(1, n_fc + 1)):
        layer = fc_layers[i - 1]
        c = cache[f'fc{i}']
        A_prev, Z, W = c['A_prev'], c['Z'], c['W']

        if i != n_fc:
            if layer['activation'] == 'relu':
                dZ = dA * relu_derivative(Z)
            else:
                dZ = dA

        grads[f'dW{n_conv + i}'] = dZ @ A_prev.T
        grads[f'db{n_conv + i}'] = dZ.sum(axis=1, keepdims=True)
        dA = W.T @ dZ

    dA = dA.T.reshape(cache['flatten_shape'])

    for i in reversed(range(1, n_conv + 1)):
        layer = conv_layers[i - 1]
        c = cache[f'conv{i}']
        Z, conv_cache, pool_cache = c['Z'], c['conv_cache'], c['pool_cache']

        dA_act = pool_backward(dA, pool_cache)
        if layer['activation'] == 'relu':
            dZ_conv = dA_act * relu_derivative(Z)
        else:
            dZ_conv = dA_act

        dA, dW, db = conv_backward(dZ_conv, conv_cache)
        grads[f'dW{i}'] = dW
        grads[f'db{i}'] = db

    return grads

In [32]:
def accuracy(AL, Y):
    return np.mean(np.argmax(AL, axis=0) == np.argmax(Y, axis=0))

In [33]:
def evaluate(X, Y, params, conv_layers, fc_layers, batch_size=128):
    """Run forward pass in batches, return (loss, accuracy) averaged over N."""
    N = X.shape[0]
    total_loss, total_correct = 0.0, 0
    for i in range(0, N, batch_size):
        Xb, Yb = X[i:i + batch_size], Y[:, i:i + batch_size]
        AL, _ = forward_pass(Xb, params, conv_layers, fc_layers)
        total_loss    += cross_entropy_loss(AL, Yb) * Xb.shape[0]
        total_correct += np.sum(np.argmax(AL, axis=0) == np.argmax(Yb, axis=0))
    return total_loss / N, total_correct / N

In [34]:
rng = np.random.default_rng(42)

conv_layers = []
conv_layers.append({
    'type': 'conv',
    'filters': 6,
    'f': 5,
    's': 1,
    'p': 0,
    'activation': 'relu',
    'pool': {
        'f': 2,
        's': 2,
        'type': 'max'
    }
}) # conv 1
conv_layers.append({
    'type': 'conv',
    'filters': 16,
    'f': 5,
    's': 1,
    'p': 0,
    'activation': 'relu',
    'pool': {
        'f': 2,
        's': 2,
        'type': 'max'
    }
}) # conv 2

fc_layers = []
fc_layers.append({
    'type': 'fc',
    'n': 120,
    'activation': 'relu'
}) # fc 1
fc_layers.append({
    'type': 'fc',
    'n': 10,
    'activation': 'softmax'
}) # fc 2

In [35]:
X_train = load_image(f'{BASE_PATH}/train-images-idx3-ubyte') # (samples, C, W, H)
Y_train = one_hot_encode(load_label(f'{BASE_PATH}/train-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (num_samples, num_classes)

X_dev = load_image(f'{BASE_PATH}/t10k-images-idx3-ubyte')  # (samples, C, W, H)
Y_dev = one_hot_encode(load_label(f'{BASE_PATH}/t10k-labels-idx1-ubyte'), num_classes=10).T # One-hot encode (num_samples, num_classes)

X_train = X_train / 255.0  # Normalize pixel values to [0, 1]
X_dev = X_dev / 255.0 

In [36]:
params = {}

# Convolutional layer parameters initialization
h, w, ch = X_train.shape[2], X_train.shape[3], X_train.shape[1]  # height, width, channels
for i, layer in enumerate(conv_layers, start=1):
    f, s, p, filters = layer['f'], layer['s'], layer['p'], layer['filters']
    h, w = output_shape(h, f, s, p), output_shape(w, f, s, p)

    filter_size = f * f * ch
    std = np.sqrt(2.0 / filter_size)  # He init (conv is followed by ReLU)
    params[f'W{i}'] = rng.standard_normal((filters, ch, f, f)) * std
    params[f'b{i}'] = np.zeros((filters, 1))

    pf, ps = layer['pool']['f'], layer['pool']['s']
    h, w = output_shape(h, pf, ps, 0), output_shape(w, pf, ps, 0)

    ch = filters  # next layer channel count is the number of filters in this layer

output_size = h * w * ch  # Flattened size after conv layers

# Fully connected layer parameters initialization
for i, layer in enumerate(fc_layers, start=1):
    n, act = layer['n'], layer['activation']
    std = np.sqrt(2.0 / output_size) if act == 'relu' else np.sqrt(1.0 / output_size)  # He init for ReLU, Xavier for softmax
    params[f'W{len(conv_layers) + i}'] = rng.standard_normal((n, output_size)) * std
    params[f'b{len(conv_layers) + i}'] = np.zeros((n, 1))
    output_size = n  # next layer input

In [ ]:
epochs = 10
mini_batch_size = 64
learning_rate = 0.01
log_every = 50       # print training stats every N mini-batches

history = {'train_loss': [], 'train_acc': [], 'dev_loss': [], 'dev_acc': []}

for epoch in range(1, epochs + 1):
    mini_batches = split_mini_batches(X_train, Y_train, mini_batch_size)
    running_loss, running_correct, running_n = 0.0, 0, 0

    for step, (X_mini, Y_mini) in enumerate(mini_batches, start=1):
        # forward
        AL, cache = forward_pass(X_mini, params, conv_layers, fc_layers)
        loss      = cross_entropy_loss(AL, Y_mini)

        # backward
        grads = backward_pass(AL, Y_mini, cache, conv_layers, fc_layers)

        # SGD update
        for k in params:
            params[k] -= learning_rate * grads['d' + k]

        # running stats for this epoch
        bs = X_mini.shape[0]
        running_loss    += loss * bs
        running_correct += np.sum(np.argmax(AL, axis=0) == np.argmax(Y_mini, axis=0))
        running_n       += bs

        if step % log_every == 0 or step == len(mini_batches):
            print(f'  epoch {epoch:3d}  step {step:4d}/{len(mini_batches)}  '
                  f'loss={running_loss / running_n:.4f}  '
                  f'acc={running_correct / running_n:.4f}')

    # end-of-epoch: training stats + dev evaluation
    train_loss = running_loss / running_n
    train_acc  = running_correct / running_n
    dev_loss, dev_acc = evaluate(X_dev, Y_dev, params, conv_layers, fc_layers)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['dev_loss'].append(dev_loss)
    history['dev_acc'].append(dev_acc)

    print(f'[epoch {epoch:3d}]  '
          f'train loss={train_loss:.4f} acc={train_acc:.4f}  |  '
          f'dev loss={dev_loss:.4f} acc={dev_acc:.4f}')

  epoch   1  step   50/938  loss=2.1235  acc=0.2503
  epoch   1  step  100/938  loss=1.8440  acc=0.3945
  epoch   1  step  150/938  loss=1.6103  acc=0.4625
  epoch   1  step  200/938  loss=1.4524  acc=0.5095
  epoch   1  step  250/938  loss=1.3503  acc=0.5414
  epoch   1  step  300/938  loss=1.2646  acc=0.5673
  epoch   1  step  350/938  loss=1.2012  acc=0.5860
  epoch   1  step  400/938  loss=1.1521  acc=0.6004
  epoch   1  step  450/938  loss=1.1077  acc=0.6148
  epoch   1  step  500/938  loss=1.0708  acc=0.6256
  epoch   1  step  550/938  loss=1.0381  acc=0.6355
  epoch   1  step  600/938  loss=1.0110  acc=0.6435
  epoch   1  step  650/938  loss=0.9857  acc=0.6506
  epoch   1  step  700/938  loss=0.9639  acc=0.6571
  epoch   1  step  750/938  loss=0.9451  acc=0.6633
  epoch   1  step  800/938  loss=0.9287  acc=0.6685
  epoch   1  step  850/938  loss=0.9141  acc=0.6738
  epoch   1  step  900/938  loss=0.9008  acc=0.6776
  epoch   1  step  938/938  loss=0.8901  acc=0.6808
[epoch   1] 